# 第二课｜数字硬件怎样保存 0.22？

上一课我们用 **漏电积分发放模型（Leaky Integrate-and-Fire, LIF）** 写出了第一个可运行神经元。公式里出现了 `0.9`、`0.22`、`1.0` 这样的数。

在 Python 里，我们几乎可以把它们当成普通小数来使用。可是当我们准备把模型搬到真实数字硬件时，一个新问题出现了：

> **一个只有有限硬件资源的数字系统，到底怎样表示带小数的数？**

本课的主要新概念：**有限位宽数值表示（finite-width numerical representation）**。

## 1. 先纠正一个常见直觉：Python float 也不是“无限精度实数”

上一课为了专注神经元模型，我们没有讨论数字在电脑里怎样保存。现在需要补上。

Python 中常用的 `float` 通常采用 64 位二进制浮点格式。它很灵活，但仍然只有有限位数，并不能精确表示所有实数。比如十进制 `0.1` 在二进制浮点中通常只能保存一个非常接近它的近似值。

所以这一课并不是“从完美实数进入不完美硬件”。更准确地说，我们要比较两种有限精度的表示方法：

- **浮点数（floating-point number）**：小数点位置可以随着指数移动，数值范围很大；
- **定点数（fixed-point number）**：小数点位置固定，表示规则更简单、更容易精确控制硬件成本。

FPGA 当然也可以做浮点运算，但很多神经形态/加速器设计会认真评估是否需要它，因为固定的小位宽定点运算往往更省资源、更容易并行。

## 2. 什么是 bit？为什么“位宽”重要？

**二进制位（binary digit, bit）** 是数字系统最基本的信息单位，通常只有两个状态：`0` 或 `1`。

多个 bit 放在一起就能表示更多状态。比如：

- 1 bit：2 种状态；
- 2 bits：4 种状态；
- 8 bits：256 种不同编码。

一个数使用多少 bit 来保存，叫它的**位宽（bit width）**。

位宽越大，通常可以表示更大的范围或更细的精度，但也会消耗更多存储和计算资源。

这就是本课真正要建立的硬件直觉：

> **数值格式不是背景细节，而是架构选择的一部分。**

## 3. 定点数的直觉：一把刻度固定的尺

把定点数想象成一把尺子。

假设我们规定：所有数都以 `1/16 = 0.0625` 为最小刻度。

那么：

- `0.0` 可以表示；
- `0.0625` 可以表示；
- `0.125` 可以表示；
- `0.1875` 可以表示；
- 但 `0.10` 不能精确落在刻度上，只能选附近某个刻度。

这里出现三个新词：

- **量化（quantization）**：把原来的数映射到有限的可表示刻度；
- **舍入（rounding）**：当它落在两个刻度之间时，决定选哪个；
- **精度（precision）**：刻度有多细。

定点数的核心不是“没有小数”，而是**小数点的位置预先固定**。

## 4. `total_bits` 和 `frac_bits` 是什么？

我们暂时不用容易产生不同约定的 `Qm.n` 记法，而直接记录两个参数：

- `total_bits`：总位宽，一共用多少 bit；
- `frac_bits`：其中多少 bit 用来表示小数精度。

如果 `frac_bits = 4`，那么缩放因子就是：

`scale = 2^4 = 16`

编码时可以把真实数乘以 16，再存成整数。例如：

`0.25 × 16 = 4`

所以整数编码 `4` 可以代表实际值 `0.25`。

如果数值不是刚好落在格点上，就需要 rounding。

In [ ]:
def quantize(x, total_bits=8, frac_bits=4):
    scale = 1 << frac_bits

    # signed integer range for this width
    min_i = -(1 << (total_bits - 1))
    max_i = (1 << (total_bits - 1)) - 1

    integer_code = round(x * scale)
    integer_code = max(min_i, min(max_i, integer_code))

    represented_value = integer_code / scale
    return integer_code, represented_value

for x in [0.1, 0.22, 0.9, 1.7, -0.3]:
    code, value = quantize(x, total_bits=8, frac_bits=4)
    print(f'{x:>5} -> integer code {code:>4} -> represented value {value}')

## 5. Observe：量化到底改变了什么？

请先看 `0.1` 和 `0.22`。

它们经过 4 个 fractional bits 后，很可能不再是原来的数，而变成最接近的可表示刻度。

请回答：

1. 这个格式的最小刻度是多少？
2. `0.22` 最终被表示成多少？误差是多少？
3. 如果把 `frac_bits` 增大，刻度会更粗还是更细？
4. 如果总位宽不变但 fractional bits 越多，可表示的最大绝对值会怎样变化？

这里已经出现一个经典工程权衡：**范围（range）和精度（precision）会竞争有限的 bit。**

## 6. 什么是 overflow？为什么 saturation 规则必须写清楚？

如果计算结果超出了当前位宽能够表示的范围，就发生了**溢出（overflow）**。

溢出以后并不存在唯一的“自然处理方式”。常见选择包括：

- **饱和（saturation）**：超过最大值就停在最大值，低于最小值就停在最小值；
- **回绕（wraparound）**：只保留有限 bit，数值可能从最大正数绕到负数。

对于膜电位或突触累积，这两个规则可能产生完全不同的神经活动。

所以硬件数值设计不能只写“用 8 bit”，还必须说明 rounding 和 overflow policy。

In [ ]:
def signed_limits(total_bits):
    return -(1 << (total_bits - 1)), (1 << (total_bits - 1)) - 1

for bits in [4, 8, 12]:
    lo, hi = signed_limits(bits)
    print(f'{bits:2d} signed bits -> integer code range [{lo}, {hi}]')

## 7. 把有限位宽重新放回 LIF 神经元

现在我们让上一课的 LIF 每一步都遵守同一套定点规则。

注意：这段代码不是最终硬件实现，而是在 Python 里建立一个**未来硬件应该遵守的数值参考模型**。

In [ ]:
def qvalue(x, total_bits, frac_bits):
    return quantize(x, total_bits, frac_bits)[1]

def run_lif_quantized(inputs, total_bits, frac_bits, alpha=0.9, threshold=1.0, reset=0.0):
    q = lambda x: qvalue(x, total_bits, frac_bits)

    v = q(0.0)
    spikes = []
    trace = []

    q_alpha = q(alpha)
    q_threshold = q(threshold)
    q_reset = q(reset)

    for t, current in enumerate(inputs):
        q_current = q(current)
        candidate_v = q(q_alpha * v + q_current)
        spike = candidate_v >= q_threshold
        v = q_reset if spike else candidate_v

        if spike:
            spikes.append(t)
        trace.append(v)

    return spikes, trace

inputs = [0.22] * 30

for fmt in [(8, 4), (12, 8), (16, 12)]:
    spikes, trace = run_lif_quantized(inputs, *fmt)
    print(f'total_bits={fmt[0]:2d}, frac_bits={fmt[1]:2d} -> spikes {spikes}')

## 8. 为什么一点数值误差可能改变 spike timing？

如果膜电位离 threshold 很远，一个很小的量化误差可能无所谓。

但如果膜电位就在 threshold 附近，`0.99` 和 `1.01` 的差别可能意味着：

- 一个实现这一时刻 spike；
- 另一个实现下一时刻才 spike；
- 两条轨迹从此开始分叉。

所以我们以后比较 CPU/Python/FPGA 时，不只看“数值平均误差很小”，还要看**spike sequence 是否保持一致或在可接受范围内**。

## 9. Try It：一次只改变一个设计参数

请做三个实验，每次先预测：

1. 固定 `total_bits=8`，比较 `frac_bits=2, 4, 6`；
2. 固定 `frac_bits=4`，比较 `total_bits=6, 8, 12`；
3. 把输入增大到可能造成 overflow 的范围，观察 saturation。

记录：

- 可表示范围；
- 最小刻度；
- spike times；
- 与上一课 floating-point 版本的差异。

## 10. AI Task

可以让 AI 帮你：

- 自动扫描不同位宽组合；
- 生成误差表；
- 画出 floating-point 与 fixed-point 膜电位轨迹对比；
- 找到“最小位宽但 spike sequence 仍保持一致”的候选方案。

要求 AI 对每个结果同时报告：`total_bits`、`frac_bits`、rounding rule、overflow rule。不要接受只写“Q8”这种含糊描述。

## 11. Human Check

不用 AI，你应该能解释：

- bit 是什么？位宽为什么是有限资源？
- floating point 和 fixed point 的核心差别是什么？
- quantization 与 rounding 分别是什么？
- overflow 为什么必须有明确规则？
- saturation 和 wraparound 为什么可能让神经网络产生完全不同的行为？
- 为什么两个数值轨迹“误差很小”，仍然可能出现不同 spike timing？

## 12. Engineering Handoff

本课成熟后的正式参考实现将进入：

`python/reference/lif_fixed.py`

选定的数值格式、rounding 和 overflow 规则会进入 MDD/TDD，成为未来 RTL 的契约。

**寄存器传输级（Register-Transfer Level, RTL）** 是我们以后描述数字硬件的一种设计层次；今天只需要知道这个名字，不需要会写 RTL。

## 13. 项目追踪 Project Trace

- Lesson ID: `LSN-002`
- Engineering slice: `RMD-002`
- Related product design: `FR1 / FR2`
- Initial tests: `T-005 ~ T-006`

这些是项目追踪信息，不是本课要背的术语。

## 14. Exit Ticket

进入下一课之前，你应该能够：

1. 展开并解释 bit、floating point、fixed point、quantization、rounding、overflow、saturation；
2. 给定 `total_bits` 和 `frac_bits`，说出最小刻度的大致大小；
3. 解释范围和精度之间的权衡；
4. 用实验说明位宽变化怎样影响 LIF 的 spike timing；
5. 明白“数值格式”本身就是硬件设计的一部分。

下一课我们要面对一个更隐蔽的问题：

> 如果两个人都说自己实现了 LIF，但一个用 `>=`、另一个用 `>`，他们实现的是同一个模型吗？